# Pancancer enrichment analysis step 2: Find enriched pathways
# Version 2: Using Reactome's Analysis Service API for enrichment analysis

## Setup

In [1]:
import cptac
import cptac.utils as ut
import pandas as pd
import numpy as np
import gseapy as gp
import os
import datetime

In [2]:
TIME_START = datetime.datetime.now().strftime('%Y%m%d_%H%M%S')

STEP1_DIR = "step1_outputs"
STEP1_FILE = "tumor_change_20200529_195104_10000_perm.tsv"
STEP1_FILE_PATH = os.path.join(STEP1_DIR, STEP1_FILE)

STEP2_DIR = "step2_outputs"
STEP2_FILE = f"urls_{TIME_START}_from_{STEP1_FILE.rsplit('.', maxsplit=1)[0]}.tsv"
STEP2_FILE_PATH = os.path.join(STEP2_DIR, STEP2_FILE)

if not os.path.isdir(STEP2_DIR):
    os.mkdir(STEP2_DIR)

## Prepare data

In [3]:
# Read in the file from step 1
data = pd.read_csv(STEP1_FILE_PATH, sep='\t', index_col=0)

# The ranked enrichment analysis service wants ranking scores
# where "bigger is better", such as expression values or
# t scores. We are ranking by adjusted p-values, where smaller
# is better. So, we'll create a column of (1 - adj_p) to use
# for ranking.
data = data.assign(one_minus_adj_p=1 - data["adj_p"])

# Make a column where all increases are +1 and all decreases 
# are -1, because these are ratioed abundances, so we can't 
# compare magnitudes between genes--we can only compare whether 
# there was a change, and whether it was positive or negative
data = data.assign(simplified_change=data["change"])

data.loc[data["change"] > 0, "simplified_change"] = 1
data.loc[data["change"] < 0, "simplified_change"] = -1
data.loc[data["change"] == 0, "simplified_change"] = 0

In [4]:
data.head()

,cancer_type,protein,change,P_val,adj_p,reject_null,protein_str,simplified_change
0,ccrcc,"('A1BG', 'NP_570602.2')",0.282928,0.0,0.0,True,A1BG,1.0
1,ccrcc,"('A1CF', 'NP_620310.1')",-0.551358,0.0,0.0,True,A1CF,-1.0
2,ccrcc,"('A2M', 'NP_000005.2')",0.595512,0.0,0.0,True,A2M,1.0
4,ccrcc,"('AAAS', 'NP_056480.1')",0.173579,0.0,0.0,True,AAAS,1.0
5,ccrcc,"('AACS', 'NP_076417.2')",-0.215967,0.0,0.0,True,AACS,-1.0


## Run enrichment analysis

In [5]:
# For each cancer, find enriched pathways for genes that changed (either up or down)   
all_enrichments = pd.DataFrame()

for cancer_type in data["cancer_type"].unique():
    
    ranked_data = data.loc[data["cancer_type"] == cancer_type, ["protein_str", "one_minus_adj_p"]].\
        set_index("protein_str").\
        rename(columns={"simplified_change": f"{cancer_type}_simplified_change"})

    analysis_token, cancer_enriched = ut.reactome_enrichment_analysis(
        analysis_type="ranked", 
        data=ranked_data, 
        sort_by="ENTITIES_FDR", 
        ascending=True,
        include_high_level_diagrams=False, 
        include_interactors=False)
    
    all_enrichments = all_enrichments.append(cancer_enriched)

In [6]:
all_enrichments.head()

,stId,name,entities_ratio,entities_pValue,entities_fdr,entities_found,entities_total
0,R-HSA-73857,RNA Polymerase II Transcription,0.116577,1.000000,1.000000,923,1692
1,R-HSA-212436,Generic Transcription Pathway,0.107000,1.000000,1.000000,829,1553
2,R-HSA-9675108,Nervous system development,0.042717,0.107131,0.795884,453,620
3,R-HSA-418555,G alpha (s) signalling events,0.041684,1.000000,1.000000,76,605
4,R-HSA-418594,G alpha (i) signalling events,0.039135,1.000000,1.000000,215,568


## Summarize enrichment results from all cancers

In [7]:
# Make a table with a list of all pathways, and:
# - The number of cancer types for which that cancer type is enriched
# - The average of the adjusted p-values for that pathway
# - The average overlap proportion for that pathway
enrichment_summary = all_enrichments[["stId", "name"]].drop_duplicates(keep="first")

times_enriched = enrichment_summary["stId"].apply(
    lambda x: all_enrichments[all_enrichments["stId"] == x].shape[0])

avg_fdr = enrichment_summary["stId"].apply(
    lambda x: all_enrichments.loc[all_enrichments["stId"] == x, "entities_fdr"].mean())

avg_overlap_props = enrichment_summary["stId"].apply(
    lambda x: all_enrichments.loc[all_enrichments["stId"] == x, "entities_ratio"].mean())

enrichment_summary = enrichment_summary.assign(
    times_enriched=times_enriched,
    avg_fdr=avg_fdr,
    avg_overlap_prop=avg_overlap_props)

enrichment_summary = enrichment_summary.sort_values(
    by=["times_enriched", "avg_overlap_prop"],
    ascending=[False, False])

enrichment_summary = enrichment_summary.reset_index(drop=True)

In [8]:
enrichment_summary.head()

,stId,name,times_enriched,avg_fdr,avg_overlap_prop
0,R-HSA-73857,RNA Polymerase II Transcription,8,0.997261,0.116577
1,R-HSA-212436,Generic Transcription Pathway,8,0.999938,0.107000
2,R-HSA-9675108,Nervous system development,8,0.447146,0.042717
3,R-HSA-418555,G alpha (s) signalling events,8,1.000000,0.041684
4,R-HSA-418594,G alpha (i) signalling events,8,1.000000,0.039135


## Visualize pathways with expression change

In [9]:
# Visualize selected pathways (this will show which genes are up or down)
to_viz = enrichment_summary[["stId", "name"]].iloc[0:4, :]

cancer_types_list = []
names_list = []
pathways_list = []
urls_list = []

for idx in to_viz.index:
    pathway = to_viz.loc[idx, "stId"]
    name = to_viz.loc[idx, "name"]
    
    for cancer_type in data["cancer_type"].unique():
        omics = data.loc[data["cancer_type"] == cancer_type, "simplified_change"]
        omics.name = f"{cancer_type}_change_tumor_normal"
        url = ut.reactome_pathway_overlay(
            pathway=pathway,
            analysis_token=analysis_tokens[cancer_type],
            open_browser=False)
        
        cancer_types_list.append(cancer_type)
        names_list.append(name)
        pathways_list.append(pathway)
        urls_list.append(url)
        

In [10]:
# Create a dataframe of the visualization URLs
urls_df = pd.DataFrame({
    "cancer_type": cancer_types_list,
    "pathway_name": names_list,
    "pathway_id": pathways_list,
    "url": urls_list})

In [11]:
# Print the dataframe in such a way that the links are clickable
urls_df.style.format('<a href="{0}">{0}</a>', subset="url")

,cancer_type,pathway_name,pathway_id,url
0,ccrcc,RNA Polymerase II Transcription,R-HSA-73857,https://reactome.org/PathwayBrowser/#/R-HSA-73857&DTAB=AN&ANALYSIS=MjAyMDA2MDMxOTAxMjBfMTkxNjM%3D
1,colon,RNA Polymerase II Transcription,R-HSA-73857,https://reactome.org/PathwayBrowser/#/R-HSA-73857&DTAB=AN&ANALYSIS=MjAyMDA2MDMxOTAzMDNfMTkxNjU%3D
2,endometrial,RNA Polymerase II Transcription,R-HSA-73857,https://reactome.org/PathwayBrowser/#/R-HSA-73857&DTAB=AN&ANALYSIS=MjAyMDA2MDMxOTA0NDNfMTkxNjc%3D
3,gbm,RNA Polymerase II Transcription,R-HSA-73857,https://reactome.org/PathwayBrowser/#/R-HSA-73857&DTAB=AN&ANALYSIS=MjAyMDA2MDMxOTA2MjdfMTkxNjk%3D
4,hnscc,RNA Polymerase II Transcription,R-HSA-73857,https://reactome.org/PathwayBrowser/#/R-HSA-73857&DTAB=AN&ANALYSIS=MjAyMDA2MDMxOTA4MTNfMTkxNzM%3D
5,lscc,RNA Polymerase II Transcription,R-HSA-73857,https://reactome.org/PathwayBrowser/#/R-HSA-73857&DTAB=AN&ANALYSIS=MjAyMDA2MDMxOTEwMDBfMTkxNzQ%3D
6,luad,RNA Polymerase II Transcription,R-HSA-73857,https://reactome.org/PathwayBrowser/#/R-HSA-73857&DTAB=AN&ANALYSIS=MjAyMDA2MDMxOTExNDVfMTkxNzc%3D
7,ovarian,RNA Polymerase II Transcription,R-HSA-73857,https://reactome.org/PathwayBrowser/#/R-HSA-73857&DTAB=AN&ANALYSIS=MjAyMDA2MDMxOTEzMzFfMTkxNzg%3D
8,ccrcc,Generic Transcription Pathway,R-HSA-212436,https://reactome.org/PathwayBrowser/#/R-HSA-212436&DTAB=AN&ANALYSIS=MjAyMDA2MDMxOTAxMjBfMTkxNjM%3D
9,colon,Generic Transcription Pathway,R-HSA-212436,https://reactome.org/PathwayBrowser/#/R-HSA-212436&DTAB=AN&ANALYSIS=MjAyMDA2MDMxOTAzMDNfMTkxNjU%3D


In [12]:
# Save the results
urls_df.to_csv(STEP2_FILE_PATH, sep='\t')

In [13]:
import webbrowser
for url in urls_df["url"][0:8]:
    webbrowser.open(url)